# Notebook 07 — EKF 2D Tracking Example

**Learning objectives**
- Apply EKF predict/update to 2D position tracking.
- Use velocity controls as model inputs.
- Visualize covariance growth (predict) and shrinking (update).
- Tune `Q` and `R` for smooth vs responsive estimates.

**Prerequisites**
- Notebook 06 EKF core ideas.


## Table of Contents

- [1) Model setup](#1-model-setup)
- [2) Exercise: smooth vs responsive tuning](#2-exercise-smooth-vs-responsive-tuning)

---

## Navigation

- Previous: [ntbk-06-ekf-core-ideas-predict-update.ipynb](./ntbk-06-ekf-core-ideas-predict-update.ipynb)
- Next: [ntbk-08-ekf-consistency-nis-nees.ipynb](./ntbk-08-ekf-consistency-nis-nees.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/ekf_slam.py`
- `src/kiss_slam/math_utils.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 106
set_seed(SEED)
plt.rcdefaults()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from notebooks._notebook_utils import plot_cov_ellipse

np.random.seed(21)


## 1) Model setup

State is 2D position: $x_k=[p_x, p_y]^T$.

Process model with velocity control $u_k=[v_x, v_y]^T$ and timestep $\Delta t$:
\[
x_k = x_{k-1} + \Delta t\,u_k + w_k
\]

Measurement model (direct noisy position):
\[
z_k = x_k + v_k
\]

Both models are linear here, so Jacobians are identity matrices.
We still write EKF-style code because EKF-SLAM uses the same structure.


In [ ]:
class EKFTracker2D:
    def __init__(self, x0, P0, Q, R):
        self.x = np.asarray(x0, dtype=float).reshape(2)
        self.P = np.asarray(P0, dtype=float).reshape(2, 2)
        self.Q = np.asarray(Q, dtype=float).reshape(2, 2)
        self.R = np.asarray(R, dtype=float).reshape(2, 2)

    def predict(self, u, dt):
        F = np.eye(2)  # x_k = x_{k-1} + dt*u
        self.x = self.x + dt * np.asarray(u, dtype=float).reshape(2)
        self.P = F @ self.P @ F.T + self.Q

    def update(self, z):
        H = np.eye(2)
        z = np.asarray(z, dtype=float).reshape(2)
        y = z - self.x
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        self.P = (np.eye(2) - K @ H) @ self.P
        return y, S, K


In [ ]:
# Simulate truth with changing controls.
N, dt = 120, 0.1
x_true = np.zeros((N, 2))
controls = np.zeros((N, 2))

for k in range(N):
    t = k * dt
    controls[k] = np.array([1.0 + 0.4*np.cos(0.25*t), 0.3*np.sin(0.7*t)])
    if k > 0:
        x_true[k] = x_true[k-1] + dt * controls[k]

meas_sigma = 0.35
z = x_true + np.random.randn(N, 2) * meas_sigma

tracker = EKFTracker2D(
    x0=[-1.5, 0.5],
    P0=np.diag([1.0, 1.0]),
    Q=np.diag([0.01, 0.01]),
    R=np.diag([meas_sigma**2, meas_sigma**2]),
)

x_est = np.zeros_like(x_true)
P_hist = np.zeros((N, 2, 2))
for k in range(N):
    tracker.predict(controls[k], dt)
    tracker.update(z[k])
    x_est[k] = tracker.x
    P_hist[k] = tracker.P


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x_true[:,0], x_true[:,1], 'k--', linewidth=2, label='truth')
ax.plot(z[:,0], z[:,1], '.', alpha=0.25, label='measurements')
ax.plot(x_est[:,0], x_est[:,1], color='tab:blue', label='EKF estimate')

for k in range(0, N, 10):
    plot_cov_ellipse(x_est[k], P_hist[k], ax=ax, n_std=2.0, edgecolor='tab:blue', alpha=0.5)

ax.set_title('2D tracking with EKF and covariance ellipses')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.axis('equal')
ax.legend(loc='best')
plt.tight_layout()


In [ ]:
# Show covariance size trend over time (trace = sum of variances).
trace_P = np.trace(P_hist, axis1=1, axis2=2)

plt.figure(figsize=(8, 3))
plt.plot(trace_P)
plt.title('Covariance trace over time')
plt.xlabel('time step')
plt.ylabel('tr(P)')
plt.grid(alpha=0.3)
plt.tight_layout()


## 2) Exercise: smooth vs responsive tuning

Try changing one setting at a time and rerun:

- `Q = diag([0.001, 0.001])` (less process uncertainty)
- `Q = diag([0.08, 0.08])` (more process uncertainty)
- `R = diag([0.05**2, 0.05**2])` (very trusted sensor)
- `R = diag([0.8**2, 0.8**2])` (noisier sensor)

Watch how the path and ellipse sizes change.

<details>
<summary><b>Optional interpretation</b></summary>

- Larger `Q` usually makes the filter react faster to measurements (more responsive).
- Larger `R` makes updates weaker, so trajectories are smoother but can lag.
- If `Q` and `R` are both too small, filter can become overconfident and brittle.

</details>


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
